# AI Optimized gsplat Training (Backend-Equivalent)

This notebook runs the same backend gsplat engine in modified + core_ai_optimization mode,
without depending on backend API orchestration.


In [ ]:
from pathlib import Path
import json
import logging
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'bimba3d_backend').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

IMAGE_DIR = Path(r'E:\Thesis\tests\b5740a69-5358-4702-a6d6-1bc57147e0a1\images_resized')
COLMAP_DIR = Path(r'E:\Thesis\tests\b5740a69-5358-4702-a6d6-1bc57147e0a1\outputs\sparse')
OUTPUT_DIR = Path(r'E:\Thesis\tests\test_jupyter_notebook\outputs\ai_optimized')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PARAMS = {
    'mode': 'modified',
    'tune_scope': 'core_ai_optimization',
    'max_steps': 15000,
    'eval_interval': 1000,
    'save_interval': 2500,
    'densify_from_iter': 500,
    'densify_until_iter': 10000,
    'densification_interval': 100,
    'densify_grad_threshold': 0.0002,
    'opacity_threshold': 0.005,
    'lambda_dssim': 0.2,
    'log_interval': 100,
    'tune_start_step': 100,
    'tune_end_step': 15000,
    'tune_interval': 100,
    'tune_min_improvement': 0.005,
    'auto_early_stop': True,
    'early_stop_monitor_interval': 200,
    'early_stop_decision_points': 10,
    'early_stop_min_eval_points': 6,
    'early_stop_min_step_ratio': 0.25,
    'early_stop_monitor_min_relative_improvement': 0.0015,
    'early_stop_eval_min_relative_improvement': 0.003,
    'early_stop_max_volatility_ratio': 0.01,
    'early_stop_ema_alpha': 0.1,
    'splat_export_interval': 1000,
    'best_splat_interval': 100,
    'use_cuda': True,
    'run_id': 'ai_notebook_run_001',
}

print('IMAGE_DIR  :', IMAGE_DIR)
print('COLMAP_DIR :', COLMAP_DIR)
print('OUTPUT_DIR :', OUTPUT_DIR)
print('mode/tune_scope:', PARAMS['mode'], PARAMS['tune_scope'])


## 2. Import Backend Engine and Shared Helpers

In [ ]:
from bimba3d_backend.worker.engines.gsplat_engine import run_training
from bimba3d_backend.worker.entrypoint import (
    _get_engine_output_dir,
    _materialize_eval_previews,
    _export_with_gsplat,
    _parse_step_from_name,
    _collect_eval_history,
    _write_json_atomic,
)

logger = logging.getLogger('notebook-gsplat-ai')
if not logger.handlers:
    logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')


## 3. Build Independent Context (No Backend Server Needed)

In [ ]:
def update_status(project_dir, status, **kwargs):
    stage = kwargs.get('stage')
    msg = kwargs.get('message')
    prog = kwargs.get('progress')
    print(f'[STATUS] {status} stage={stage} progress={prog} msg={msg}')

def write_metrics(project_dir, payload, **kwargs):
    target = Path(project_dir) / 'metrics.json'
    target.parent.mkdir(parents=True, exist_ok=True)
    with open(target, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2)

context = {
    'logger': logger,
    'update_status': update_status,
    'write_metrics': write_metrics,
    'get_engine_output_dir': _get_engine_output_dir,
    'materialize_eval_previews': _materialize_eval_previews,
    'export_with_gsplat': _export_with_gsplat,
    'parse_step_from_name': _parse_step_from_name,
    'collect_eval_history': _collect_eval_history,
    'write_json_atomic': _write_json_atomic,
}


## 4. Train with Exact Backend Engine (Modified AI Mode)

In [ ]:
# Notebook-safe worker settings on Windows to avoid multiprocessing spawn issues.
os.environ['GSPLAT_NUM_WORKERS'] = '0'
os.environ['BIMBA3D_DOCKER_WORKER'] = '1'

result = run_training(
    image_dir=IMAGE_DIR,
    colmap_dir=COLMAP_DIR,
    output_dir=OUTPUT_DIR,
    params=PARAMS,
    resume=False,
    context=context,
)

print('Training finished. Return:', result)


## 5. Inspect Output Artifacts

In [ ]:
engine_dir = OUTPUT_DIR / 'engines' / 'gsplat'
print('Engine output dir:', engine_dir)
if engine_dir.exists():
    for p in sorted(engine_dir.glob('*')):
        if p.is_file():
            print('-', p.name, p.stat().st_size, 'bytes')
        else:
            print('-', p.name, '(dir)')

adaptive_file = engine_dir / 'adaptive_tuning_results.json'
if adaptive_file.exists():
    print('Found adaptive tuning report:', adaptive_file.name)
